# Vector算子章节实践：Sigmoid算子开发

通过前面的学习，我们已经掌握了使用AscendC开发矢量算子的完整方法。本实践要求实现 Sigmoid 算子，并且完成精度验证。

Sigmoid 计算公式：
```text
sigmoid(x) = 1 / (1 + exp(-x))
```

实践要求：
- 完成`Sigmoid`的Kernel侧计算逻辑。
- 在Host侧直接完成泛化Tiling计算。
- 在Host侧使用 `sigmoid_custom<<<...>>>` 直接调用核函数。
- 支持 float/half 类型输入输出。
- 支持以下调用case并运行成功：
  - case1：输入 x 的 shape 为 `(7, 83)`，类型为 float32。
  - case2：输入 x 的 shape 为 `(1024, 1024)`，类型为 float16。
  - case3：输入 x 的 shape 为 `(999, 999)`，类型为 float32。

请开始你的实践，体验完整开发过程。

---
## 环境准备

In [ ]:
!mkdir -p Sources/02.15

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("Environment initialization process completed successfully!")

---
## 目录结构介绍

本次实践代码按照如下目录结构存放：

```
├── 2.15
│   ├── scripts
│   │   ├── gen_data.py                // 输入数据和真值数据生成脚本
│   │   └── verify_result.py           // 验证输出数据和真值数据是否一致的验证脚本
│   ├── CMakeLists.txt                 // 编译工程文件
│   ├── data_utils.h                   // 数据读入写出函数
│   ├── sigmoid_custom.asc             // host侧调用 + kernel侧实现
│   └── run.sh                         // 运行脚本
```

In [ ]:
!cp -rf ./src/02.15 ./Sources

---
## Sigmoid 直调实现
该实现将 Host 侧 Tiling、Kernel 侧 Sigmoid 计算、`<<<>>>` 调用以及三组 case 验证放在同一个 `sigmoid_custom.asc` 文件中。
请填充下列TODO内容。

In [ ]:
%%writefile Sources/02.15/sigmoid_custom.asc

#include <algorithm>
#include <cmath>
#include <cstdint>
#include <iostream>
#include <type_traits>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#include "tiling/platform/platform_ascendc.h"
#include "data_utils.h"

constexpr int32_t BUFFER_NUM = 2;

struct SigmoidCustomTilingData {
    uint32_t smallCoreDataNum;
    uint32_t bigCoreDataNum;
    uint32_t finalBigTileNum;
    uint32_t finalSmallTileNum;
    uint32_t tileDataNum;
    uint32_t smallTailDataNum;
    uint32_t bigTailDataNum;
    uint32_t tailBlockNum;
};

int64_t GetShapeSize(const std::vector<int64_t> &shape)
{
    int64_t shapeSize = 1;
    for (auto dim : shape) {
        shapeSize *= dim;
    }
    return shapeSize;
}

template <typename T>
void CalcTiling(uint32_t inputNum, SigmoidCustomTilingData &tiling, uint32_t &numBlocks)
{
    // TODO: 将numBlocks和tiling结构体进行刷新
}

template <typename T>
 __global__ __vector__ void sigmoid_custom(GM_ADDR x, GM_ADDR y, SigmoidCustomTilingData tiling)
{
    // TODO: 填充Sigmoid的计算逻辑
}

template <typename T>
void RunCase(const char *caseName, const std::vector<int64_t> &shape, int caseIndex)
{
    printf("Testcase: %s start ======\n", caseName);

    const uint32_t inputNum = static_cast<uint32_t>(GetShapeSize(shape));
    size_t dataSize = inputNum * sizeof(T);
    std::string srcFile = "./input/src_gm_case" + std::to_string(caseIndex) + ".bin";
    std::string dstFile = "./output/output_case" + std::to_string(caseIndex) + ".bin";

    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    SigmoidCustomTilingData tiling{};
    uint32_t numBlocks = 0;
    CalcTiling<T>(inputNum, tiling, numBlocks);

    uint8_t* xHost;
    uint8_t *xDevice;
    aclrtMallocHost((void**)(&xHost), dataSize);
    aclrtMalloc((void**)&xDevice, dataSize, ACL_MEM_MALLOC_HUGE_FIRST);
    ReadFile(srcFile, dataSize, xHost, dataSize);
    aclrtMemcpy(xDevice, dataSize, xHost, dataSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t *yHost;
    uint8_t *yDevice;
    aclrtMallocHost(reinterpret_cast<void **>(&yHost), dataSize);
    aclrtMalloc(reinterpret_cast<void **>(&yDevice), dataSize, ACL_MEM_MALLOC_HUGE_FIRST);

    sigmoid_custom<T><<<numBlocks, nullptr, stream>>>(xDevice, yDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(yHost, dataSize, yDevice, dataSize, ACL_MEMCPY_DEVICE_TO_HOST);
    WriteFile(dstFile, yHost, dataSize);

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFreeHost(xHost);
    aclrtFreeHost(yHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
}

int32_t main(int32_t argc, char *argv[])
{
    RunCase<float>("case0 float32 shape=(7,83)", {7, 83}, 0);
    RunCase<half>("case1 float16 shape=(1024,1024)", {1024, 1024}, 1);
    RunCase<float>("case2 float32 shape=(999,999)", {999, 999}, 2);
    return 0;
}

---
## 编译运行
运行后应看到三组 case 均输出 `test pass`。

In [ ]:
!cd Sources/02.15 && bash run.sh

---
## 查看答案
实现方式不唯一，这里给出的答案仅供参考。

In [ ]:
!cat ./answer/02.15_answer/sigmoid_custom.asc